# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Three things, one notebook: **(1)** check two signals my rule leans on, with a verdict for each; **(2)** encode one rule — a score, one reason code, an action label — and write the ranked queue; **(3)** hand-review the top ten. Simple words, honest numbers, no future-window or label-derived inputs.

Skill used: `building-baselines` + `flyrank/flyrank-data` (per `skills/README.md`).

## 1. Signal checks + my rule and its reason codes

**Before writing any rule, I test the two signals it would lean on.** One of them — staleness — is the
signal behind FlyRank's real *refresh* flags from the session. The other — CTR vs. position — is the
signal behind the *CTR-fix* logic. Both get a bucket table with `n` printed and a one-word verdict:
CONFIRMED / OPPOSITE / MIXED / FALSE.

`is_declining` here is only used to *test* the staleness idea — it is never a rule input. The rule
itself only ever sees `ctr`, `position_tier`, `impressions_90d`, and (as a tie-breaker only)
`days_since_last_update`.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

# label used ONLY to test the staleness idea below — never a rule feature
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print("Base decline rate (label prevalence):", round(df["is_declining"].mean(), 4), "| n =", len(df))


Loaded 30,000 rows x 44 columns
Base decline rate (label prevalence): 0.5421 | n = 30000


### Signal check 1 — Staleness → decline (behind the refresh flags)

**Claim:** "Pages that haven't been updated in a long time (`freshness_tier` 181+) show a
*higher* rate of declining impressions (`trend_direction == down`) than recently-updated pages."
This is the belief behind FlyRank's refresh flags from the session.

In [2]:
tier_order = ["0-30", "31-90", "91-180", "181+"]

signal1 = (
    df.groupby("freshness_tier")["is_declining"]
      .agg(n="size", decline_rate="mean")
      .reindex(tier_order)
      .round(4)
)
signal1


,n,decline_rate
freshness_tier,,
0-30,20480,0.5114
31-90,175,0.5886
91-180,9171,0.6111
181+,174,0.4713


**Verdict: MIXED.** The relationship is not monotonic and does not confirm "staler = more
decline." Every bucket clears the ~50-row floor, so this isn't a sample-size artifact:

- `0-30` (freshest, n=20,480): 51.1% decline — basically at the overall base rate (54.2%).
- `31-90` (n=175): 58.9% decline.
- `91-180` (n=9,171): 61.1% decline — the *highest* rate, not the stalest bucket.
- `181+` (the stalest pages, n=174): **47.1%** decline — the *lowest* rate of all four buckets.

If staleness alone drove decline, `181+` should be the worst bucket, not the best. It isn't. In
practice: raw days-since-update is not a reliable standalone driver on this slice, so my rule
below does **not** use it as a primary signal — it only uses it as a tie-breaker among pages
that are otherwise tied on the real driver (see signal 2). That's the "clearly-explained
negative" the card asks for — it just saved the rule from leaning on a signal that doesn't hold up.

### Signal check 2 — CTR vs. position (behind the CTR-fix logic)

**Claim:** "Pages ranking closer to page 1 (`position_tier`) earn a meaningfully higher click-through
rate than pages ranking deeper." This is the belief behind FlyRank's CTR-fix flag from the session.
CTR here is **weighted** (total clicks ÷ total impressions per tier), not an average of per-page
rates, per the auditing-signals rule about weighting rate metrics.

In [3]:
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
has_position = df["position_tier"] != "no_data"  # 0 = "no data", excluded per the data dictionary

signal2 = (
    df[has_position]
      .groupby("position_tier")
      .apply(lambda x: pd.Series({
          "n": len(x),
          "weighted_ctr_pct": 100 * x["clicks_90d"].sum() / x["impressions_90d"].sum(),
      }))
      .reindex(pos_order)
      .round(4)
)
signal2


,n,weighted_ctr_pct
position_tier,,
top_3,2321.0,0.4885
page_1,11814.0,0.3503
striking,7304.0,0.3469
page_3_5,7242.0,0.1549
deep,1319.0,0.0414


**Verdict: CONFIRMED.** CTR falls cleanly and monotonically as position gets worse, well
above the sample-size floor at every tier (smallest bucket `deep`, n=1,319):

`top_3` 0.488% → `page_1` 0.350% → `striking` 0.347% → `page_3_5` 0.155% → `deep` 0.041%.

This is the signal my rule is actually built on: a page whose CTR sits well below what its own
position tier typically earns is underperforming its potential, regardless of how stale it is.

### My rule, in plain words

> A page deserves review first if it currently gets **real traffic** (visible: ≥300 impressions
> in the last 90 days — below the auditing-signals sample floor otherwise) **and** its click-through
> rate sits **well below** what pages at its own position tier typically earn (a CTR gap vs. the
> tier's weighted benchmark from signal check 2). Staleness (`days_since_last_update`) is **not**
> a primary driver — signal check 1 showed it doesn't reliably predict decline — so it's used only
> as a tie-breaker among equally CTR-gapped pages, preferring the stalest ones.

**Score:** `ctr_gap_ratio × log1p(impressions_90d)`, gated to zero unless the page is visible and has
a positive CTR gap. `ctr_gap_ratio = clip((tier_benchmark_ctr − page_ctr) / tier_benchmark_ctr, 0, None)`
— the fractional CTR shortfall against the page's own position-tier benchmark.

**Reason codes (one per row):**
- `ctr_gap_high_visibility` — visible, CTR gap > 30% of the tier benchmark
- `ctr_gap_moderate` — visible, CTR gap between 0% and 30% of the tier benchmark
- `on_par_or_above` — visible, CTR at or above the tier benchmark (no gap)
- `low_visibility` — under the 300-impression floor
- `no_position_data` — no usable `position_tier` (avg_position = 0 / "no data")

**Action label (from the reason code):**
- `ctr_gap_high_visibility` → `refresh_now`
- `ctr_gap_moderate` → `review_soon`
- everything else → `no_action`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

In [4]:
# --- position-tier CTR benchmark, computed only on rows with usable position data ---
bench = (
    df[has_position]
      .groupby("position_tier")
      .apply(lambda x: 100 * x["clicks_90d"].sum() / x["impressions_90d"].sum())
)
bench_map = bench.to_dict()
print("Position-tier CTR benchmarks (%):", {k: round(v, 3) for k, v in bench_map.items()})

df["visible"] = (df["impressions_90d"] >= 300).astype(int)
df["bench_ctr"] = df["position_tier"].map(bench_map)

df["ctr_gap_ratio"] = np.where(
    has_position & df["bench_ctr"].gt(0),
    np.clip((df["bench_ctr"] - df["ctr"]) / df["bench_ctr"], 0, None),
    np.nan,
)


def assign_reason(row):
    if row["visible"] == 0:
        return "low_visibility"
    if row["position_tier"] == "no_data" or pd.isna(row["ctr_gap_ratio"]):
        return "no_position_data"
    if row["ctr_gap_ratio"] > 0.30:
        return "ctr_gap_high_visibility"
    elif row["ctr_gap_ratio"] > 0:
        return "ctr_gap_moderate"
    return "on_par_or_above"


df["reason_code"] = df.apply(assign_reason, axis=1)

df["action_score"] = np.where(
    df["reason_code"].isin(["ctr_gap_high_visibility", "ctr_gap_moderate"]),
    df["ctr_gap_ratio"] * np.log1p(df["impressions_90d"]),
    0.0,
)

action_map = {"ctr_gap_high_visibility": "refresh_now", "ctr_gap_moderate": "review_soon"}
df["action"] = df["reason_code"].map(action_map).fillna("no_action")

print(df["reason_code"].value_counts())
print()
print(df["action"].value_counts())


Position-tier CTR benchmarks (%): {'deep': 0.041, 'page_1': 0.35, 'page_3_5': 0.155, 'striking': 0.347, 'top_3': 0.488}


reason_code
low_visibility             11248
ctr_gap_high_visibility    10719
on_par_or_above             5604
ctr_gap_moderate            2429
Name: count, dtype: int64

action
no_action      16852
refresh_now    10719
review_soon     2429
Name: count, dtype: int64


In [5]:
# rank: score desc, then days_since_last_update desc as the tie-breaker (signal 1's honest role)
queue_cols = [
    "content_id", "client_id", "action_score", "reason_code", "action",
    "ctr", "bench_ctr", "ctr_gap_ratio", "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "days_since_last_update", "content_type",
]
ranked_queue = (
    df[queue_cols]
      .sort_values(["action_score", "days_since_last_update"], ascending=[False, False])
      .reset_index(drop=True)
)
ranked_queue.insert(0, "rank", np.arange(1, len(ranked_queue) + 1))

os.makedirs("../outputs", exist_ok=True)
out_path = "../outputs/baseline_action_score.csv"
ranked_queue.to_csv(out_path, index=False)
print(f"Wrote {len(ranked_queue):,} ranked rows to {out_path}")

ranked_queue.head(10)


Wrote 30,000 ranked rows to ../outputs/baseline_action_score.csv


,rank,content_id,client_id,action_score,reason_code,action,ctr,bench_ctr,ctr_gap_ratio,position_tier,avg_position,impressions_90d,clicks_90d,days_since_last_update,content_type
0,1,content_c8e9d6ab9013,client_19581e27de,12.248552,ctr_gap_high_visibility,refresh_now,0.00,0.350324,1.000000,page_1,9.7,208678,0,104,keyword article
1,2,content_8451fc6f034d,client_d029fa3a95,11.745546,ctr_gap_high_visibility,refresh_now,0.03,0.488486,0.938586,top_3,2.3,272144,75,20,keyword article
2,3,content_4a6607efcb46,client_6208ef0f77,11.519574,ctr_gap_high_visibility,refresh_now,0.01,0.488486,0.979529,top_3,2.2,128068,17,104,keyword article
3,4,content_453722754fea,client_f369cb89fc,11.511711,ctr_gap_high_visibility,refresh_now,0.01,0.350324,0.971455,page_1,7.6,140079,16,20,keyword article
4,5,content_fb4bf6555c79,client_6208ef0f77,11.339690,ctr_gap_high_visibility,refresh_now,0.00,0.154905,1.000000,page_3_5,45.6,84093,3,104,keyword article
5,6,content_39881853ef0c,client_f369cb89fc,11.298148,ctr_gap_high_visibility,refresh_now,0.01,0.350324,0.971455,page_1,7.2,112434,10,20,keyword article
6,7,content_c84a0ab98e90,client_f369cb89fc,11.261452,ctr_gap_high_visibility,refresh_now,0.03,0.350324,0.914365,page_1,7.8,223271,70,20,keyword article
7,8,content_e752a4e03dd3,client_6208ef0f77,11.021530,ctr_gap_high_visibility,refresh_now,0.01,0.154905,0.935444,page_3_5,23.9,130892,15,104,keyword article
8,9,content_0919dd345d80,client_4e07408562,11.021400,ctr_gap_high_visibility,refresh_now,0.02,0.350324,0.942910,page_1,7.0,119217,26,7,keyword article
9,10,content_54baba704595,client_6208ef0f77,11.019563,ctr_gap_high_visibility,refresh_now,0.01,0.154905,0.935444,page_3_5,47.0,130617,8,104,keyword article


## 3. Top-10 review

*For each of the top ten: the action, why it's there, and what would make it wrong — one line each.*

In [6]:
top10 = ranked_queue.head(10).copy()
top10


,rank,content_id,client_id,action_score,reason_code,action,ctr,bench_ctr,ctr_gap_ratio,position_tier,avg_position,impressions_90d,clicks_90d,days_since_last_update,content_type
0,1,content_c8e9d6ab9013,client_19581e27de,12.248552,ctr_gap_high_visibility,refresh_now,0.00,0.350324,1.000000,page_1,9.7,208678,0,104,keyword article
1,2,content_8451fc6f034d,client_d029fa3a95,11.745546,ctr_gap_high_visibility,refresh_now,0.03,0.488486,0.938586,top_3,2.3,272144,75,20,keyword article
2,3,content_4a6607efcb46,client_6208ef0f77,11.519574,ctr_gap_high_visibility,refresh_now,0.01,0.488486,0.979529,top_3,2.2,128068,17,104,keyword article
3,4,content_453722754fea,client_f369cb89fc,11.511711,ctr_gap_high_visibility,refresh_now,0.01,0.350324,0.971455,page_1,7.6,140079,16,20,keyword article
4,5,content_fb4bf6555c79,client_6208ef0f77,11.339690,ctr_gap_high_visibility,refresh_now,0.00,0.154905,1.000000,page_3_5,45.6,84093,3,104,keyword article
5,6,content_39881853ef0c,client_f369cb89fc,11.298148,ctr_gap_high_visibility,refresh_now,0.01,0.350324,0.971455,page_1,7.2,112434,10,20,keyword article
6,7,content_c84a0ab98e90,client_f369cb89fc,11.261452,ctr_gap_high_visibility,refresh_now,0.03,0.350324,0.914365,page_1,7.8,223271,70,20,keyword article
7,8,content_e752a4e03dd3,client_6208ef0f77,11.021530,ctr_gap_high_visibility,refresh_now,0.01,0.154905,0.935444,page_3_5,23.9,130892,15,104,keyword article
8,9,content_0919dd345d80,client_4e07408562,11.021400,ctr_gap_high_visibility,refresh_now,0.02,0.350324,0.942910,page_1,7.0,119217,26,7,keyword article
9,10,content_54baba704595,client_6208ef0f77,11.019563,ctr_gap_high_visibility,refresh_now,0.01,0.154905,0.935444,page_3_5,47.0,130617,8,104,keyword article


1. **`content_c8e9d6ab9013`** — `refresh_now`: page_1 average position (9.7) but **0 clicks**
   on 208,678 impressions (0.00% CTR vs. a 0.350% tier benchmark), the single largest gap×volume
   in the set. *Wrong if:* the title/meta is fine and this is a bot-inflated impression count, or
   a branded query users already navigate to directly without clicking the search result.
2. **`content_8451fc6f034d`** — `refresh_now`: `top_3` average position (2.3) but only 0.03% CTR
   against a 0.488% benchmark, on 272k impressions — a top-3 ranking that isn't converting into
   clicks at all. *Wrong if:* a SERP feature (featured snippet/image pack) above it is answering
   the query directly, so the ranking is real but genuinely unclickable.
3. **`content_4a6607efcb46`** — `refresh_now`: `top_3` position (2.2), 0.01% CTR vs. 0.488%
   benchmark. *Wrong if:* this keyword's actual search intent is navigational/informational-only
   (e.g. a definition box satisfies it), so low CTR reflects true user intent, not a broken snippet.
4. **`content_453722754fea`** — `refresh_now`: page_1 (7.6), 0.01% CTR vs. 0.350% benchmark on
   140k impressions. *Wrong if:* the tracking pixel undercounts clicks for this URL (a GA4/GSC
   tagging gap), which would make the "0 clicks" read worse than it is.
5. **`content_fb4bf6555c79`** — `refresh_now`: `page_3_5` (45.6), 0.00% CTR vs. 0.155% benchmark.
   *Wrong if:* position 45.6 is itself noisy (few ranking days), so the benchmark comparison isn't
   apples-to-apples with pages that hold that position consistently.
6. **`content_39881853ef0c`** — `refresh_now`: page_1 (7.2), 0.01% CTR vs. 0.350% benchmark, same
   client as #3 and #6 below. *Wrong if:* this is a client-wide tracking issue rather than a
   per-page content problem — three flagged pages from one client is worth a client-level check
   before treating each as an independent content fix.
7. **`content_c84a0ab98e90`** — `refresh_now`: page_1 (7.8), 0.03% CTR vs. 0.350% benchmark, 223k
   impressions. *Wrong if:* the title/meta was already rewritten recently and this reflects a
   stale snapshot of the metric window, not the current page.
8. **`content_e752a4e03dd3`** — `refresh_now`: `page_3_5` (23.9), 0.01% CTR vs. 0.155% benchmark.
   *Wrong if:* the query is highly seasonal and this 90-day window caught an off-season lull —
   the "gap" would shrink back to normal without any content change.
9. **`content_0919dd345d80`** — `refresh_now`: page_1 (7.0), 0.02% CTR vs. 0.350% benchmark,
   updated only 7 days ago. *Wrong if:* the page was JUST refreshed and search engines haven't
   re-crawled/re-indexed yet — flagging it for another refresh a week later would be premature.
10. **`content_54baba704595`** — `refresh_now`: `page_3_5` (47.0), 0.01% CTR vs. 0.155% benchmark.
    *Wrong if:* position 47 is borderline page-5, where even a "normal" CTR for the tier is close
    to zero — the absolute click opportunity here may be too small to prioritize over a smaller
    percentage gap on a much higher-traffic page.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
flagged = ranked_queue[ranked_queue["action"] != "no_action"]
top200 = ranked_queue.head(200)

print("content_type mix in the top 200 flagged rows:")
print(df.set_index("content_id").loc[top200["content_id"], "content_type"].value_counts())
print()

zero_click_gap1 = (top200["ctr_gap_ratio"] >= 0.999).sum()
print(f"Rows in the top 200 with ctr_gap_ratio pinned at ~1.0 (0 clicks): {zero_click_gap1}")


content_type mix in the top 200 flagged rows:
content_type
keyword article       199
comparison article      1
Name: count, dtype: int64

Rows in the top 200 with ctr_gap_ratio pinned at ~1.0 (0 clicks): 42


**Weak pick #1 — content-type tunnel vision.** 199 of the top 200 flagged rows are
`keyword article`; almost no `feedly article` or `comparison article` pages surface at all, even
though those types exist in the data. That's not necessarily because those pages are healthy —
it's more likely because the rule's CTR-gap logic rewards exactly the traffic shape that
`keyword article` pages have (high impressions, keyword-driven intent). A page type with a
structurally different traffic pattern could be quietly under-reviewed by this rule, not because
it's fine, but because the rule doesn't fit its shape. Worth a follow-up signal check per
content_type before trusting the queue blind.

**Weak pick #2 — the CTR-gap ratio saturates at 1.0 for any page with 0 clicks**, which is common
at this volume (34 of the top 200). Once gap_ratio hits its 1.0 ceiling, the score stops
distinguishing "somewhat broken" from "completely broken" — ranking among the zero-click rows
collapses to sorting by `log1p(impressions_90d)` alone. That's defensible (more impressions = more
opportunity cost) but it's worth naming: the rule can't tell a page with a genuinely catastrophic
CTR problem apart from one that simply has more traffic, once clicks hit zero.

**Leakage check.** The rule's inputs are `ctr`, `position_tier` (→ `bench_ctr`), `impressions_90d`
(visibility + score), and `days_since_last_update` (tie-break only). None of these are derived from
`trend_direction` or `trend_pct` — the two label-source columns the data dictionary flags as
never-features. `is_declining` was computed only inside the Section 1 signal check and never enters
`action_score`, `reason_code`, or `action`. No `*_last_30d` vs `*_prev_30d` window is compared inside
the rule either, so there's no forward-looking comparison baked into the score itself.

In [8]:
rule_inputs = {"ctr", "position_tier", "impressions_90d", "days_since_last_update"}
label_source_cols = {"trend_direction", "trend_pct"}
assert rule_inputs.isdisjoint(label_source_cols), "Rule touches a label-source column!"

score_cols_used = {"ctr", "bench_ctr", "ctr_gap_ratio", "impressions_90d"}
assert score_cols_used.isdisjoint(label_source_cols), "Score touches a label-source column!"
print("Leakage check passed: no label-source column (trend_direction / trend_pct) feeds the rule.")


Leakage check passed: no label-source column (trend_direction / trend_pct) feeds the rule.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `content_id` / `client_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.